In [ ]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader

import fluidbench
from fluidbench.cfdataset import CFDataset

In [ ]:
ID = "cylinder_Re41_interp"
dataset_path = fluidbench.download(ID)

In [ ]:
def load_split(split):
    folder = dataset_path / split
    cases = sorted((path for path in folder.glob("*.h5") if path.stem != "constants"), key=lambda path: int(path.stem))
    data = []
    for case in cases:
        with h5py.File(case, "r") as file:
            data.append((file["inputs/scalars"][:], file["inputs/fields"][:], file["targets/fields"][:]))
    return data

def to_dataset(cases):
    scalars, fields, targets = (np.concatenate(arrays) for arrays in zip(*cases))
    return CFDataset(scalars, fields, targets, np.arange(len(targets)))

In [ ]:
# Cases are stored chronologically (not shuffled); for now each split holds one case.
train_dataset = to_dataset(load_split("train"))
validation_dataset = to_dataset(load_split("validation"))
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=8, shuffle=False)
len(train_dataset), len(validation_dataset)

In [ ]:
### Model definition
from minimal_models import BaselineZero as architecture

weights_path = Path("weights") / ID / f"{architecture.__name__}.pt"

model = architecture()
model.to(model.device)

In [ ]:
history = model.fit(train_loader, validation_loader)

In [ ]:
weights_path.parent.mkdir(parents=True, exist_ok=True)
model.save_weights(weights_path)
weights_path